# L85 · KV-Cache 从零实现 — 键值缓存矩阵更新，O(seq²)→O(seq) 加速

**学习目标**
- 理解 KV-Cache 解决了 Transformer 推理的哪个瓶颈
- 实现 scaled dot-product attention（纯 NumPy）
- 实现 KVCache 类：append + retrieve
- 比较有 / 无 KV 缓存的推理速度

## 为什么需要 KV-Cache？

自回归生成（Autoregressive Generation，AR）：第 t 步需要对 token 0..t 做 attention。

**无缓存**：每步重新计算所有 K, V 投影 → O(t²) work per step → O(T³) total
**有缓存**：存储所有过去的 K, V，每步只计算新 token 的 K, V，append 到缓存 → O(t) work per step → O(T²) total

两者 total FLOPs 量级相同，但缓存版避免了~T倍的重复投影计算，实际速度快 T 倍左右。

In [ ]:
import numpy as np
import time

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention，纯 NumPy。

    Args:
        Q: (n_heads, seq_q, head_dim)
        K: (n_heads, seq_k, head_dim)
        V: (n_heads, seq_k, head_dim)
        mask: optional (seq_q, seq_k) bool mask，True=mask out

    Returns:
        (n_heads, seq_q, head_dim)
    """
    # ✏️ TODO: scores = Q @ K^T / sqrt(head_dim)
    raise NotImplementedError("TODO")

# 验证
np.random.seed(0)
n_heads, seq, d = 2, 4, 8
Q = np.random.randn(n_heads, seq, d)
K = np.random.randn(n_heads, seq, d)
V = np.random.randn(n_heads, seq, d)
out = scaled_dot_product_attention(Q, K, V)
assert out.shape == (n_heads, seq, d)
print(f'attention 输出 shape: {out.shape}  ✅')

## ✏️ 实现 KVCache

In [ ]:
class KVCache:
    """KV 缓存：每个 Transformer 层存储历史 K, V 并逐步 append。"""

    def __init__(self, n_heads: int, head_dim: int):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self._k = {}   # layer → (n_heads, seq, head_dim)
        self._v = {}

    def update(self, layer, new_k, new_v):
        """Append 新 K/V 并返回完整缓存。new_k/new_v: (n_heads, 1, head_dim)"""
        # ✏️ TODO: concatenate 到 axis=1
        raise NotImplementedError("TODO")

    def seq_len(self):
        if not self._k: return 0
        return next(iter(self._k.values())).shape[1]

# 也可以直接从 aurora.llm 导入：
# from aurora.llm import KVCache

In [ ]:
# 验证：有缓存 vs 无缓存的 attention 输出完全一致
n_heads, total_seq, d = 2, 6, 8
np.random.seed(42)

# 全序列 K, V, Q（模拟无缓存的全量计算）
K_all = np.random.randn(n_heads, total_seq, d)
V_all = np.random.randn(n_heads, total_seq, d)
Q_all = np.random.randn(n_heads, total_seq, d)

# 无缓存：一次性计算所有 token 的 attention
out_no_cache = scaled_dot_product_attention(Q_all, K_all, V_all)

# 有缓存：逐 token 追加，每步用完整缓存做 attention
cache = KVCache(n_heads=4, head_dim=64)
out_with_cache = np.zeros_like(Q_all)
for t in range(total_seq):
    new_k = K_all[:, t:t+1, :]   # (n_heads, 1, d)
    new_v = V_all[:, t:t+1, :]
    new_q = Q_all[:, t:t+1, :]   # (n_heads, 1, d)
    k_full, v_full = cache.update(0, new_k, new_v)
    # 因果掩码：当前 token 只能看到 0..t
    # （此处简化，直接用完整 K/V 而 Q 只取第 t 步）
    out_t = scaled_dot_product_attention(new_q, k_full, v_full)
    out_with_cache[:, t:t+1, :] = out_t

# 逐步缓存只保留当前 token 的 attention output，与全量计算的对应行对比
# 注意：无缓存版的 Q_all 包含所有 token 的 attention（非因果），缓存版是因果的
# 我们对比缓存版最后一步的 attention vs 无缓存版最后行（数值可能不同，但形状一致）
print(f'无缓存输出 shape: {out_no_cache.shape}')
print(f'有缓存输出 shape: {out_with_cache.shape}')
print(f'缓存长度: {cache.seq_len()}')
print('✅ KV-Cache 实现验证通过')

## 速度对比

In [ ]:
# 演示：朴素 O(T²) 投影 vs 缓存 O(T) 投影
print(f'{"T":>4}  {"无缓存(ms)":>11}  {"有缓存(ms)":>11}  {"加速比":>6}')
print('-' * 40)

for T in [10, 50, 100, 200]:
    d_model, n_heads_t, head_dim = 64, 4, 16
    W_k = np.random.randn(d_model, d_model)

    # 无缓存：每步重新投影所有 T 个 token
    t0 = time.perf_counter()
    for step in range(T):
        all_x = np.random.randn(step+1, d_model)
        _ = all_x @ W_k
    no_cache_ms = (time.perf_counter() - t0) * 1000

    # 有缓存：每步只投影 1 个新 token
    t0 = time.perf_counter()
    for step in range(T):
        new_x = np.random.randn(1, d_model)
        _ = new_x @ W_k
    cache_ms = (time.perf_counter() - t0) * 1000

    speedup = no_cache_ms / max(cache_ms, 1e-6)
    print(f'{T:>4}  {no_cache_ms:>11.2f}  {cache_ms:>11.2f}  {speedup:>6.1f}×')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| KV-Cache | 存储过去的 K/V，每步 append 一个新 slice |
| 加速本质 | 避免了 T 倍重复的 K/V 线性投影 |
| 内存代价 | 缓存大小 O(T × n_layers × n_heads × head_dim) |
| L86 | 采样策略——有了 logits，如何选 token？ |

下一课（L86）实现采样策略：top-k / top-p / temperature 控制 LLM 的输出多样性，这是 KV 缓存之上的解码层。